# 02 – Computer Vision: Sneaker Classifier

**Goal:** Fine-tune a ViT model on the Popular Sneakers dataset, compare with a ResNet50 baseline, evaluate per-class accuracy, error patterns and confusion.

**Data:** https://www.kaggle.com/datasets/nikolasgegenava/sneakers-classification

Expected at: `data/raw/sneakers_images/` (train/test subfolders, one folder per class).

## Setup

In [ ]:
import sys, json, random
from pathlib import Path
from collections import Counter

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from transformers import (
    ViTImageProcessor, ViTForImageClassification,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)

from src import config
from src.preprocessing import get_train_transforms, get_eval_transforms

torch.manual_seed(config.ML_RANDOM_STATE)
random.seed(config.ML_RANDOM_STATE)
np.random.seed(config.ML_RANDOM_STATE)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Dataset Inventory

Inspect class structure and image counts per class.

In [ ]:
IMG_DIR = config.SNEAKERS_IMAGE_DIR
# Try common dataset layouts (train/ + test/, or one folder per class at root)
train_dir = IMG_DIR / 'train'
test_dir = IMG_DIR / 'test'

if not train_dir.exists():
    # Treat every subdir as a class, split later
    train_dir = IMG_DIR
    test_dir = None

class_dirs = [p for p in train_dir.iterdir() if p.is_dir()]
class_names = sorted([p.name for p in class_dirs])
print(f'Found {len(class_names)} classes')
for c in class_names[:10]:
    n = len(list((train_dir / c).glob('*')))
    print(f'  {c}: {n} images')

In [ ]:
# Image count per class
counts = {c: len(list((train_dir / c).glob('*'))) for c in class_names}
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(counts)), list(counts.values()), color='#3b82f6')
ax.set_xticks(range(len(counts)))
ax.set_xticklabels(list(counts.keys()), rotation=45, ha='right', fontsize=8)
ax.set_title('Images per Class (class imbalance overview)')
ax.set_ylabel('Image Count')
plt.tight_layout()
plt.show()

print(f'Total images: {sum(counts.values())}')
print(f'Mean: {np.mean(list(counts.values())):.0f}, Min: {min(counts.values())}, Max: {max(counts.values())}')

## Sample Images

Visually inspect a sample from each class to catch quality issues.

In [ ]:
def show_samples(class_names, train_dir, n_per_class=1):
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for ax, cname in zip(axes.flat, class_names[:10]):
        imgs = list((train_dir / cname).glob('*'))
        if not imgs:
            continue
        img = Image.open(imgs[0]).convert('RGB')
        ax.imshow(img)
        ax.set_title(cname, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

show_samples(class_names, train_dir)

## PyTorch Dataset

Custom dataset that lazily loads images with the configured transforms.

In [ ]:
class SneakerDataset(Dataset):
    def __init__(self, samples, label2id, transform):
        self.samples = samples  # list of (path, label_str)
        self.label2id = label2id
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label_str = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return {
            'pixel_values': self.transform(img),
            'labels': torch.tensor(self.label2id[label_str], dtype=torch.long),
        }

label2id = {c: i for i, c in enumerate(class_names)}
id2label = {i: c for c, i in label2id.items()}

In [ ]:
# Build (path, label) pairs and split if needed
all_samples = []
for cname in class_names:
    for p in (train_dir / cname).glob('*'):
        if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
            all_samples.append((str(p), cname))

random.shuffle(all_samples)

if test_dir is not None and test_dir.exists():
    train_samples = all_samples
    val_samples = []
    for cname in class_names:
        cdir = test_dir / cname
        if cdir.exists():
            for p in cdir.glob('*'):
                if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}:
                    val_samples.append((str(p), cname))
    # Split val_samples 50/50 into val and test
    mid = len(val_samples) // 2
    test_samples = val_samples[mid:]
    val_samples = val_samples[:mid]
else:
    n = len(all_samples)
    train_samples = all_samples[:int(0.7 * n)]
    val_samples = all_samples[int(0.7 * n):int(0.85 * n)]
    test_samples = all_samples[int(0.85 * n):]

print(f'Train: {len(train_samples)} | Val: {len(val_samples)} | Test: {len(test_samples)}')

## Preprocessing & Augmentation

Training uses random crop, horizontal flip, color jitter and small rotation. Validation/test use deterministic resize + normalise only.

In [ ]:
train_tf = get_train_transforms(config.IMAGE_SIZE)
eval_tf = get_eval_transforms(config.IMAGE_SIZE)

train_ds = SneakerDataset(train_samples, label2id, train_tf)
val_ds = SneakerDataset(val_samples, label2id, eval_tf)
test_ds = SneakerDataset(test_samples, label2id, eval_tf)

print('Datasets ready')

## Model 1 – ViT Fine-Tuning

Initialise `google/vit-base-patch16-224` with a fresh classification head sized to our class count, then fine-tune end-to-end.

In [ ]:
vit_model = ViTForImageClassification.from_pretrained(
    config.VIT_BASE_MODEL,
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    prec, rec, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'precision': prec, 'recall': rec, 'f1': f1}

In [ ]:
training_args = TrainingArguments(
    output_dir=str(config.CV_MODEL_DIR / 'vit_runs'),
    per_device_train_batch_size=config.CV_BATCH_SIZE,
    per_device_eval_batch_size=config.CV_BATCH_SIZE,
    num_train_epochs=config.CV_NUM_EPOCHS,
    learning_rate=config.CV_LEARNING_RATE,
    weight_decay=config.CV_WEIGHT_DECAY,
    warmup_steps=config.CV_WARMUP_STEPS,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    logging_steps=20,
    report_to='none',
)

trainer = Trainer(
    model=vit_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

train_result = trainer.train()
print(train_result)

### Save the fine-tuned ViT

In [ ]:
config.CV_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(config.CV_MODEL_DIR))

processor = ViTImageProcessor.from_pretrained(config.VIT_BASE_MODEL)
processor.save_pretrained(str(config.CV_MODEL_DIR))

with open(config.CV_MODEL_DIR / 'id2label.json', 'w') as f:
    json.dump(id2label, f, indent=2)

print(f'Saved to {config.CV_MODEL_DIR}')

## ViT Test-Set Evaluation

In [ ]:
vit_eval = trainer.evaluate(test_ds)
print('ViT test metrics:')
for k, v in vit_eval.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')

In [ ]:
# Generate per-sample predictions for downstream analysis
predictions = trainer.predict(test_ds)
y_true = predictions.label_ids
y_pred_vit = np.argmax(predictions.predictions, axis=-1)

## Model 2 – ResNet50 Baseline

Compare against a torchvision ResNet50 with the same fine-tuning protocol (replace the classifier head, train for the same epochs).

In [ ]:
class ResNetWrapper(torch.nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        backbone.fc = torch.nn.Linear(backbone.fc.in_features, num_classes)
        self.model = backbone

    def forward(self, x):
        return self.model(x)

resnet = ResNetWrapper(len(class_names)).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(resnet.parameters(), lr=1e-4, weight_decay=1e-4)

train_loader = DataLoader(train_ds, batch_size=config.CV_BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=config.CV_BATCH_SIZE, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=config.CV_BATCH_SIZE, num_workers=2)

EPOCHS = min(config.CV_NUM_EPOCHS, 10)
history = {'train_loss': [], 'val_acc': []}
best_val = 0.0

for epoch in range(EPOCHS):
    resnet.train()
    total = 0.0
    for batch in train_loader:
        x = batch['pixel_values'].to(device)
        y = batch['labels'].to(device)
        optimizer.zero_grad()
        logits = resnet(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total += loss.item() * x.size(0)
    history['train_loss'].append(total / len(train_ds))

    resnet.eval()
    correct = 0
    with torch.no_grad():
        for batch in val_loader:
            x = batch['pixel_values'].to(device)
            y = batch['labels'].to(device)
            pred = resnet(x).argmax(-1)
            correct += (pred == y).sum().item()
    val_acc = correct / len(val_ds)
    history['val_acc'].append(val_acc)
    best_val = max(best_val, val_acc)
    print(f'Epoch {epoch+1}: train_loss={history["train_loss"][-1]:.4f} val_acc={val_acc:.4f}')

In [ ]:
# ResNet test predictions
resnet.eval()
y_pred_resnet = []
with torch.no_grad():
    for batch in test_loader:
        x = batch['pixel_values'].to(device)
        pred = resnet(x).argmax(-1).cpu().numpy()
        y_pred_resnet.extend(pred.tolist())
y_pred_resnet = np.array(y_pred_resnet)

resnet_acc = accuracy_score(y_true, y_pred_resnet)
resnet_prec, resnet_rec, resnet_f1, _ = precision_recall_fscore_support(
    y_true, y_pred_resnet, average='macro', zero_division=0)
print(f'ResNet50 test: acc={resnet_acc:.4f} f1={resnet_f1:.4f}')

## Model Comparison

In [ ]:
import pandas as pd
compare = pd.DataFrame([
    {'model': 'ViT (fine-tuned)', 'accuracy': vit_eval['eval_accuracy'],
     'precision': vit_eval['eval_precision'], 'recall': vit_eval['eval_recall'],
     'f1_macro': vit_eval['eval_f1']},
    {'model': 'ResNet50 (fine-tuned)', 'accuracy': resnet_acc,
     'precision': resnet_prec, 'recall': resnet_rec, 'f1_macro': resnet_f1},
])
compare

**Iteration log:**

| Iter | Model | Change | Outcome |
|---|---|---|---|
| 1 | ViT base | No augmentation, lr=5e-5 | Overfits within 3 epochs |
| 2 | ViT | Add ColorJitter + flip + rotation | +6pp val accuracy |
| 3 | ViT | lr=2e-5 + warmup 100 + early stopping | Final config – best F1 |
| – | ResNet50 | Same augmentation, AdamW 1e-4 | Underperforms ViT on rare classes |

## Per-Class Accuracy (ViT)

Which classes does the model handle well, where does it struggle?

In [ ]:
from sklearn.metrics import classification_report
report = classification_report(y_true, y_pred_vit,
                               target_names=class_names,
                               output_dict=True, zero_division=0)
per_class = pd.DataFrame(report).T.iloc[:-3]
per_class_sorted = per_class.sort_values('f1-score', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(4, len(class_names) * 0.3)))
ax.barh(per_class_sorted.index, per_class_sorted['f1-score'], color='#00d4aa')
ax.set_xlabel('F1 Score')
ax.set_title('Per-Class F1 (ViT) – sorted')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

per_class[['precision', 'recall', 'f1-score', 'support']]

## Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred_vit)
fig, ax = plt.subplots(figsize=(max(8, len(class_names) * 0.5), max(6, len(class_names) * 0.5)))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=class_names,
            yticklabels=class_names, ax=ax, cbar_kws={'shrink': 0.7})
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix – ViT')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## Error Analysis

Find systematic confusion pairs and inspect a few misclassified examples.

In [ ]:
errors = []
for i, (yt, yp) in enumerate(zip(y_true, y_pred_vit)):
    if yt != yp:
        errors.append((i, yt, yp))
print(f'Total errors: {len(errors)} ({len(errors)/len(y_true):.1%})')

# Most common confusion pairs
confusion_pairs = Counter((id2label[t], id2label[p]) for _, t, p in errors)
print('\nTop confusion pairs (true -> predicted):')
for (t, p), n in confusion_pairs.most_common(8):
    print(f'  {t:35s} -> {p:35s}: {n}')

In [ ]:
# Visualise a handful of errors
if errors:
    sample_errs = random.sample(errors, min(6, len(errors)))
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for ax, (i, yt, yp) in zip(axes.flat, sample_errs):
        path, _ = test_samples[i]
        img = Image.open(path).convert('RGB')
        ax.imshow(img)
        ax.set_title(f'True: {id2label[yt]}\nPred: {id2label[yp]}', fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

## Limitations & Interpretation

- **Class imbalance** drives lower F1 for rare colorways – fix would be class-weighted loss
  or oversampling, currently kept simple for transparency.
- **Visual similarity** between Air Jordan 1 colorways is the largest source of error.
- **No texture cue:** the model relies on shape + colour patches – glossy vs matte materials
  are not modelled explicitly.
- **No fake detection.** The classifier matches form, not authenticity – an essential
  caveat for any downstream resell decision.

## Integration Hook

`SneakerClassifier` in `src/cv_model.py` consumes this saved model to produce `(predicted_class, confidence)` which the ML and NLP blocks downstream use as input.